# Tur ghidat al datelor

Acest notebook este punctul de plecare pentru orice analiză. Te conectezi la baza de date, vezi cum sunt structurate datele și înveți cele câteva reguli de care depinde corectitudinea oricărei interogări.

Dacă rulezi pentru prima oară: asigură-te că `data/streets.db` există (vezi README).

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
ACCENT = '#C04F35'
INK    = '#15171A'
MUTED  = '#6E6E70'

conn = sqlite3.connect('../data/streets.db')
conn.row_factory = sqlite3.Row
print('Connected.')

## 1. Ce e în bază

Sursa primară este Registrul Secțiilor de Vot publicat de Autoritatea Electorală Permanentă. Înainte de orice query, e util să vezi tabelele și view-urile disponibile.

In [ ]:
pd.read_sql("""
    SELECT type, name
    FROM sqlite_master
    WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%'
    ORDER BY type, name
""", conn)

## 2. Regula nr. 1 — folosește `streets_dedup`, nu `streets`

Tabelul `streets` conține o linie pe stradă **per secție de votare**. O stradă lungă care traversează cinci secții apare de cinci ori. Numărarea directă din `streets` umflă totul cu un factor variabil.

View-ul `streets_dedup` deduplică pe `(uat, name_normalized)` — exact unitatea pe care vrem să o numărăm.

In [ ]:
df = pd.read_sql("""
    SELECT
      (SELECT COUNT(*) FROM streets)         AS raw_rows,
      (SELECT COUNT(*) FROM streets_dedup)   AS deduped_rows,
      (SELECT COUNT(DISTINCT name_normalized) FROM streets_dedup) AS distinct_names,
      (SELECT COUNT(DISTINCT siruta)         FROM streets_dedup) AS uats,
      (SELECT COUNT(DISTINCT judet)          FROM streets_dedup) AS judete
""", conn)
df.T.rename(columns={0:'count'}).style.format({'count':'{:,.0f}'})

Raportul `raw_rows / deduped_rows` îți spune câte secții traversează în medie o stradă. Dacă vreodată un număr îți pare uriaș, primul lucru de verificat e dacă ai uitat `streets_dedup`.

## 3. Cele patru tabele de curare

Pe lângă datele brute, există patru tabele construite manual (sau prin LLM batch) care îmbogățesc semnificativ analiza:

| Tabel | Ce conține | Cheie de join |
|-------|------------|----------------|
| `persons` | persoane onorate — gen, profesie, epocă, QID Wikidata, sitelinks | `core_name_norm` |
| `name_categories` | clasificare semantică — abstract, ideologic, mitologie, etc. | `core_name_norm` |
| `nature_terms` | termeni din natură — flori, copaci, păsări, apă, munte | `core_name_norm` |
| `place_refs` | referințe la locuri — orașe, țări, regiuni geografice | `core_name_norm` |

Important: cheia de join este `core_name_norm`, **nu** `name_normalized`. `core_name` este partea esențială a numelui — "Mihai Eminescu" rămâne "Mihai Eminescu" indiferent dacă strada se cheamă "Strada Mihai Eminescu" sau "Bulevardul Poet Mihai Eminescu".

In [ ]:
pd.read_sql("""
    SELECT 'persons'         AS table_name, COUNT(*) AS rows FROM persons
    UNION ALL
    SELECT 'name_categories', COUNT(*) FROM name_categories
    UNION ALL
    SELECT 'nature_terms',    COUNT(*) FROM nature_terms
    UNION ALL
    SELECT 'place_refs',      COUNT(*) FROM place_refs
""", conn)

## 4. Top străzi după număr de apariții

Cele mai frecvente nume de stradă din România. Observă cum nature_terms și persons se intercalează în topul absolut.

In [ ]:
top = pd.read_sql("""
    SELECT name_normalized AS name,
           COUNT(*) AS streets,
           COUNT(DISTINCT judet) AS judete
    FROM streets_dedup
    WHERE is_numeric = 0
    GROUP BY name_normalized
    ORDER BY streets DESC
    LIMIT 20
""", conn)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top['name'][::-1], top['streets'][::-1], color=INK, alpha=0.85)
ax.set_title('Top 20 nume de stradă în România', fontsize=13, pad=10)
ax.set_xlabel('Număr de străzi (deduplicat per UAT)')
for i, (n, s) in enumerate(zip(top['streets'][::-1], top['judete'][::-1])):
    ax.text(n + 5, i, f'{n}  ·  {s} jud.', va='center', fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()

## 5. Acoperirea clasificării

Câte dintre străzi pot fi atribuite unei categorii (persoană / natură / abstract / ideologic / ...) și câte rămân neclasificate?

In [ ]:
coverage = pd.read_sql("""
    SELECT
      SUM(CASE WHEN p.core_name_norm IS NOT NULL THEN 1 ELSE 0 END) AS persons,
      SUM(CASE WHEN nc.core_name_norm IS NOT NULL THEN 1 ELSE 0 END) AS categories,
      SUM(CASE WHEN nt.core_name_norm IS NOT NULL THEN 1 ELSE 0 END) AS nature,
      SUM(CASE WHEN pr.core_name_norm IS NOT NULL THEN 1 ELSE 0 END) AS places,
      SUM(CASE WHEN sd.is_numeric = 1 THEN 1 ELSE 0 END) AS numeric_streets,
      COUNT(*) AS total
    FROM streets_dedup sd
    LEFT JOIN persons         p  ON p.core_name_norm  = sd.core_name_norm
    LEFT JOIN name_categories nc ON nc.core_name_norm = sd.core_name_norm
    LEFT JOIN nature_terms    nt ON nt.core_name_norm = sd.core_name_norm
    LEFT JOIN place_refs      pr ON pr.core_name_norm = sd.core_name_norm
""", conn)

total = coverage['total'][0]
rows = [('persoane', coverage['persons'][0]),
        ('natură', coverage['nature'][0]),
        ('categorie semantică', coverage['categories'][0]),
        ('locuri', coverage['places'][0]),
        ('numerice', coverage['numeric_streets'][0])]
for label, n in rows:
    print(f'  {label:.<24} {n:>7,} ({n/total*100:5.1f}%)'.replace(',', '.'))
print(f'  {"total străzi":.<24} {total:>7,}'.replace(',', '.'))

Notă: aceeași stradă poate apărea în mai multe categorii (un nume de persoană onorat cu o referință naturală în el ar fi rar dar posibil), deci suma procentelor nu trebuie să dea 100%.

## 6. Un join concret — top poeți după număr de străzi

Pattern-ul standard: pornești din `streets_dedup`, joini cu o tabelă curată pe `core_name_norm`, grupezi după persoană.

In [ ]:
pd.read_sql("""
    SELECT p.full_name,
           p.era,
           COUNT(*) AS streets,
           COUNT(DISTINCT sd.siruta) AS uats
    FROM streets_dedup sd
    JOIN persons p ON p.core_name_norm = sd.core_name_norm
    WHERE p.profession = 'poet'
    GROUP BY p.core_name_norm
    ORDER BY streets DESC
    LIMIT 10
""", conn)

## 7. Unde mergem mai departe

Acum că știi structura, restul notebook-urilor analizează povești specifice:

- `01_gender_gap` — ecartul de gen în onorările publice
- `02_recognition_scope` — Wikipedia sitelinks ca semnal de notorietate
- `03_ideological_names` — moștenirea comunistă în nomenclatura străzilor
- `04_nature_themes` — flori, copaci, păsări — dominanța naturii
- `05_regional_patterns` — ce face fiecare județ altfel
- `06_eminescu` — un studiu de caz: cel mai onorat poet român

Fiecare reia conexiunea, deci poți rula oricare independent.

In [ ]:
conn.close()